In [ ]:
%pip install torch torchvision transformers diffusers accelerate Pillow hf_transfer matplotlib numpy tqdm

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [ ]:
import os
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

In [ ]:
import torch

device = torch.device('cpu')
if torch.cuda.is_available():
    # CUDA 사용 가능
    device = torch.device('cuda')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    # Apple Silicon (MPS) 사용 가능
    device = torch.device('mps')
print("사용 장치:", device)

import numpy as np
import random
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# windows - cuda
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# apple silicon - mps
if torch.mps.is_available():
    torch.mps.manual_seed(SEED)
    torch.use_deterministic_algorithms(True, warn_only=True)

# Linear Probing

## 데이터셋 불러오기 및 변환

In [ ]:
import torchvision
# CIFAR-10 데이터셋 로드 (train 50,000장, test 10,000장)
# --- CIFAR-10 데이터셋 로드 ---
# root: 데이터가 저장될 경로
# train=True: 훈련용 데이터셋을 불러옴
# download=True: 해당 경로에 데이터가 없으면 다운로드
# transform: 데이터 변환기 적용 -> 데이터를 먼저 확인하기 위해 False로 설정
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=False)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=False)

print(trainset.data.shape, testset.data.shape) 

In [ ]:
 # CIFAR_10 데이터셋의 평균과 표준편차 계산
data_tensor = torch.from_numpy(trainset.data)

# 변환을 위한 평균/표준편차 계산 (배치, 높이, 너비 차원에 대해)
mean = torch.mean(data_tensor.float() / 255.0, dim=(0, 1, 2))
std = torch.std(data_tensor.float() / 255.0, dim=(0, 1, 2))

# 변환(transform)선언
# 1. (224, 224) 사이즈로 리사이즈 해주세요.
# 2. 이미지를 텐서로 변환해주세요.
# 3. 전체 학습데이터의 평균/표준편차를 계산해 이에 대해 정규화를 진행해주세요.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

In [ ]:
# root: 데이터가 저장될 경로
# train=True: 훈련용 데이터셋을 불러옴
# download=True: 해당 경로에 데이터가 없으면 다운로드
# transform: 위에서 정의한 데이터 변환기 적용
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

# DataLoader는 데이터를 미니배치(mini-batch) 단위로 묶어주는 역할을 함
# batch_size: 한 번에 모델에 입력할 데이터(이미지)의 개수
# shuffle=True: 훈련 시 데이터를 무작위로 섞어 모델이 데이터 순서에 과적합되는 것을 방지
# shuffle=False: 평가 시 데이터 순서를 고정해 재현 가능한 검증을 수행
# pin_memory=True: CPU 메모리를 고정해 GPU로 텐서를 더 빠르게 전송
# num_workers=8: 데이터 로딩에 사용할 병렬 프로세스(worker) 수
trainloader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True, pin_memory=True, num_workers=8)
testloader  = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, pin_memory=True, num_workers=8)
print("훈련 배치 개수:", len(trainloader), "테스트 배치 개수:", len(testloader))
 

## 사전학습된 ResNet-18 가져오기

In [ ]:
# ImageNet 데이터로 사전 학습된 ResNet-18 모델을 불러옴
# pretrained=True 옵션은 학습된 가중치(weights)를 함께 가져오라는 의미
model = torchvision.models.resnet18(pretrained=True) 

In [ ]:
model

## 선형 분류 계층 교체

In [ ]:
# 1. 마지막 분류층(Fully Connected Layer) 교체
# ResNet-18의 마지막 층을 수행해주세요.
# 기존 ResNet-18의 마지막 층(model.fc)은 ImageNet의 1000개 클래스를 분류합니다.
# 우리는 CIFAR-10의 10개 클래스를 분류하도록 새로운 `nn.Linear` 레이어로 교체해야 합니다.
model.fc = nn.Linear(model.fc.in_features, 10)


# 2. 선형 프로빙을 위해 ResNet-18을 1에서 선언한 마지막 층을 제외하고 업데이트가 되지 않도록 가중치를 동결하세요.
# `param.requires_grad = False`로 설정하면 해당 파라미터는 학습 중에 업데이트되지 않습니다.
for name, param in model.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

# 장치를 GPU로 이동
model = model.to(device)

## Linear Probing

In [ ]:
from tqdm import tqdm
# 1. 손실 함수 (`criterion`)로는 다중 클래스 분류를 위한 크로스 엔트로피 오차(`nn.CrossEntropyLoss`)를 사용합니다.
criterion = nn.CrossEntropyLoss()

# 2. 옵티마이저 (`optimizer`)로는 마지막 레이어 (분류층) 파라미터만 업데이트하도록 SGD(Stochastic Gradient Descent)를 사용합니다.
# 학습률은 0.001로 설정해주세요.
optimizer = optim.SGD(model.fc.parameters(), lr=0.001)

# 3. 학습을 위한 for 루프도 마무리해주세요. 이전에 진행한 챕터 1-2의 실습과 과제를 떠올려주세요.
num_epochs = 5
for epoch in range(num_epochs):
    model.train()       # 모델을 학습 모드로 설정
    running_loss = 0.0  # 에포크 동안의 총 손실을 기록할 변수

    # trainloader에서 미니배치 단위로 데이터를 가져와 반복
    for xb, yb in tqdm(trainloader):
        # 입력 데이터와 정답 레이블을 지정된 장치(GPU)로 이동
        xb, yb = xb.to(device), yb.to(device)

        # 3.1 옵티마이저의 그래디언트(gradient)를 0으로 초기화
        optimizer.zero_grad()

        # 3.2 모델에 입력을 넣어 순전파(forward pass) 진행 및 출력(outputs) 계산
        outputs = model(xb)

        # 3.3 모델의 출력과 실제 정답을 비교하여 손실(loss) 계산
        loss = criterion(outputs, yb)

        # 3.4 역전파(backward pass)를 통해 각 파라미터에 대한 그래디언트 계산
        loss.backward()

        # 3.5 옵티마이저를 사용해 모델의 파라미터(가중치)를 업데이트
        # requires_grad=True로 설정된 파라미터만 업데이트됨 (여기서는 fc 층만)
        optimizer.step()

        # 현재 배치의 손실을 running_loss에 더함
        running_loss += loss.item()

    # 전체 에포크가 끝난 후 평균 훈련 손실을 계산하고 출력
    avg_loss = running_loss / len(trainloader)
    print(f"[Epoch {epoch+1}/{num_epochs}] 평균 훈련 손실: {avg_loss:.4f}")

## 모델 평가

In [ ]:
# 모델을 평가 모드로 설정
# 이 모드에서는 드롭아웃(Dropout)이나 배치 정규화(Batch Normalization) 등이 비활성화되어 일관된 예측 결과를 얻을 수 있음
model.eval()

correct = 0   # 맞춘 예측 개수
total = 0     # 전체 데이터 개수

# 그래디언트 계산을 비활성화하는 컨텍스트
with torch.no_grad():
    # testloader에서 미니배치 단위로 데이터를 가져와 반복
    for xb, yb in tqdm(testloader):

        # 입력 데이터와 정답 레이블을 장치(GPU)로 이동
        xb, yb = xb.to(device), yb.to(device)

        # 모델에 입력을 넣어 출력 계산
        outputs = model(xb)
        _, predicted = torch.max(outputs.data, 1)
        total += yb.size(0) # 현재 배치의 데이터 개수를 total에 더함

        # 예측이 정답과 일치하는 개수를 세어 correct에 더함
        correct += (predicted == yb).sum().item()

# 전체 정확도 계산 및 출력
accuracy = 100 * correct / total
print(f"테스트 데이터 정확도 (Linear Probing): {accuracy:.2f}%")